# Lyric-Music Congruence — Pipeline

End-to-end, **resumable** pipeline. Every stage skips work that is already done and
persists progress to `data/project.db`, so you can stop and restart at any point.

**Source of truth:** the LRCLIB dump (songs with time-synced lyrics). Audio,
popularity, mood and embeddings all key off the LRCLIB track ID.

Stages: `setup → sample → audio → popularity → mood → chorus → embeddings → alignment → combine`.

Prerequisites: install `requirements.txt`, place `lrclib-db-dump-*.sqlite3` under `data/`
(or set `LRCLIB_DUMP`), and export `SPOTIFY_CLIENT_ID` / `SPOTIFY_CLIENT_SECRET`
(plus optional `YOUTUBE_API_KEY`, `LASTFM_API_KEY`).

In [2]:
import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))   # make `lmc` importable

from lmc import (config, lrclib, audio, popularity, mood,
                 chorus, embeddings, mert, genre, alignment, combine, db)
from lmc.utils import setup_logging
setup_logging(logging.INFO)

config.ensure_dirs()
print(config.summary())

Repo root     : /Users/budge.13/Desktop/Music Analysis
LRCLIB dump   : /Users/budge.13/Desktop/Music Analysis/data/lrclib-db-dump-20260604.sqlite3
Project DB    : /Users/budge.13/Desktop/Music Analysis/data/project.db
Audio dir     : /Users/budge.13/Desktop/Music Analysis/data/audio
Embeddings    : /Users/budge.13/Desktop/Music Analysis/data/embeddings
Results dir   : /Users/budge.13/Desktop/Music Analysis/results
Spotify creds : set
YouTube key   : optional/unset
Last.fm key   : optional/unset


## 1 — Setup & sample

`setup()` reports how many synced-lyric songs exist in LRCLIB (the *universe*),
how many we've already sampled, and roughly how many remain. The first call runs
a one-time full count (cached afterwards). Then pick a **session target** and
`sample()` randomly draws that many *new* songs into the corpus.

In [2]:
status = lrclib.setup()
status

21:05:25 | INFO | lmc.lrclib | LRCLIB corpus — universe 19,507,240 | sampled 550 | remaining ≈ 19,506,690


{'universe': 19507240, 'sampled': 550, 'remaining': 19506690}

In [3]:
TARGET = 1000          # how many NEW songs to add this session
new_ids = lrclib.sample(TARGET, seed=None)
print(f'Added {len(new_ids)} songs; corpus now {lrclib.setup()["sampled"]}.')

21:05:31 | INFO | lmc.lrclib | Sampled 1000 new songs (1139 attempts). Corpus now 1550.
21:05:31 | INFO | lmc.lrclib | LRCLIB corpus — universe 19,507,240 | sampled 1,550 | remaining ≈ 19,505,690


Added 1000 songs; corpus now 1550.


## 2 — Audio (official audio, not music videos)

Downloads `data/audio/<track_id>.mp3` via yt-dlp, preferring auto-generated
“- Topic” channels and audio uploads over music videos. Captures YouTube
view / like / comment counts. Pass `limit=` to cap a session.

In [4]:
audio.download_pending(limit=None)

21:05:31 | INFO | lmc.audio | Audio: 1007 songs to fetch.
21:05:31 | INFO | lmc.audio | [1/1007] Corb Lund feat. Jaida Dreyer — I Think You Oughta Try Whiskey
ERROR: [youtube] 4wnCesouSOM: This video is not available
21:05:39 | WARNING | lmc.audio |   search failed: ERROR: [youtube] 4wnCesouSOM: This video is not available
21:05:39 | INFO | lmc.audio |     ✗ not_found
21:05:39 | INFO | lmc.audio | [2/1007] Minus the Bear — Hold Me Down


21:05:50 | INFO | lmc.audio |     ✓ Zerhino (121,014 views)
21:05:50 | INFO | lmc.audio | [3/1007] Nick Drake — Time Has Told Me


21:06:01 | INFO | lmc.audio |     ✓ Nick Drake (1,750,284 views)
21:06:01 | INFO | lmc.audio | [4/1007] Fleetwood Mac — If You Let Me Love You


21:06:14 | INFO | lmc.audio |     ✓ Fleetwood Mac (26,954 views)
21:06:14 | INFO | lmc.audio | [5/1007] Björk — I've Seen It All


21:06:26 | INFO | lmc.audio |     ✓ Instrumental Music (160 views)
21:06:26 | INFO | lmc.audio | [6/1007] Cannibal Ox — Battle for Asgard (feat. L.I.F.E. Long and C-Rayz Walz)


...

23:00:03 | INFO | lmc.audio |     ✓ Big Chuck Lyric Video (13,223 views)
23:00:03 | INFO | lmc.audio | [625/1007] Vincent Delerm — Chatenay Malabry


23:00:14 | INFO | lmc.audio |     ✓ Vincent Delerm (75,615 views)
23:00:14 | INFO | lmc.audio | [626/1007] Lady Gaga feat. Beyonce — Telephone


23:00:25 | INFO | lmc.audio |     ✓ Watermelon Music (7,976,062 views)
23:00:25 | INFO | lmc.audio | [627/1007] Sandy Nelson — Teen Beat


23:00:37 | INFO | lmc.audio |     ✓ Sandy Nelson - Topic (11,120 views)
23:00:37 | INFO | lmc.audio | [628/1007] Nielson & Miss Montreal — Hoe
ERROR: [youtube] 7ECB5AigzzY: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
23:00:39 | WARNING | lmc.audio |   search failed: ERROR: [youtube] 7ECB5AigzzY: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
23:00:39 | INFO | lmc.audio |     ✗ not_found

{'attempted': 1007, 'done': 562, 'failed': 445}

## 3 — Popularity & genre recovery

Spotify popularity + artist genre tags (→ recovered genre / orientation), Deezer
rank, optional Last.fm, and the YouTube metrics mirrored from stage 2.

In [5]:
popularity.fetch_pending(limit=None)

23:14:06 | INFO | lmc.popularity | Popularity: 1000 songs to enrich.
23:14:06 | INFO | lmc.popularity | [1/1000] Corb Lund feat. Jaida Dreyer — I Think You Oughta Try Whiskey
23:14:08 | INFO | lmc.popularity |     ✓ pop=38 genre=country/narrative
23:14:08 | INFO | lmc.popularity | [2/1000] Minus the Bear — Hold Me Down
23:14:09 | INFO | lmc.popularity |     ✓ pop=29 genre=rock/narrative
23:14:09 | INFO | lmc.popularity | [3/1000] Nick Drake — Time Has Told Me
23:14:10 | INFO | lmc.popularity |     ✓ pop=53 genre=folk/narrative
23:14:10 | INFO | lmc.popularity | [4/1000] Fleetwood Mac — If You Let Me Love You
23:14:11 | INFO | lmc.popularity |     ✓ pop=16 genre=rock/narrative
23:14:11 | INFO | lmc.popularity | [5/1000] Björk — I've Seen It All
23:14:12 | INFO | lmc.popularity |     ✓ pop=34 genre=pop/production
23:14:12 | INFO | lmc.popularity | [6/1000] Cannibal Ox — Battle for Asgard (feat. L.I.F.E. Long and C-Rayz Walz)
23:14:13 | INFO | lmc.popularity |     ✓ pop=1 genre=hip-hop/na

{'attempted': 1000, 'found': 996}

## 4 — Acoustic controls (mood proxies + MERT) & 5 — Chorus detection

Two control representations are computed so the Stan models can toggle between
them (`controls = "mood" | "mert" | "both"`):

- **mood** — librosa signal-processing proxies (interpretable, cheap).
- **MERT** — one 1024-d `MERT-v1-330M` vector per song; `combine` PCA-reduces these
  to `mert_pc01..K` controls. MERT is a *different* representation than MuLan/CLAP,
  so its PCs are valid **off-LMC-path** controls (they soak up production/era/genre
  nuisance variance without partialling out LMC's inputs).

Both need audio; chorus detection is lyrics-only.

In [6]:
mood.extract_pending(limit=None)     # librosa mood proxies  (controls='mood')
mert.extract_pending(limit=None)     # MERT-v1-330M vectors  (controls='mert', default)
chorus.compute_pending()             # chorus vs non-chorus line flags

00:58:39 | INFO | lmc.mood | Mood: 562 songs to process.
00:58:39 | INFO | lmc.mood | [1/562] 65851
01:14:06 | INFO | lmc.mood | [2/562] 91431
01:14:10 | INFO | lmc.mood | [3/562] 167539
01:14:15 | INFO | lmc.mood | [4/562] 222237
01:14:19 | INFO | lmc.mood | [5/562] 281140
01:14:24 | INFO | lmc.mood | [6/562] 326218
01:14:28 | INFO | lmc.mood | [7/562] 434739
01:14:33 | INFO | lmc.mood | [8/562] 489995
01:14:37 | INFO | lmc.mood | [9/562] 531194
01:14:42 | INFO | lmc.mood | [10/562] 635544
01:14:46 | INFO | lmc.mood | [11/562] 669831
01:14:51 | INFO | lmc.mood | [12/562] 834390
01:30:44 | INFO | lmc.mood | [13/562] 895865
01:30:49 | INFO | lmc.mood | [14/562] 1003269
01:30:53 | INFO | lmc.mood | [15/562] 1050842
01:30:58 | INFO | lmc.mood | [16/562] 1074484
01:31:02 | INFO | lmc.mood | [17/562] 1147609
01:31:07 | INFO | lmc.mood | [18/562] 1152931
01:31:11 | INFO | lmc.mood | [19/562] 1175444
01:31:16 | INFO | lmc.mood | [20/562] 1196880
01:31:20 | INFO | lmc.mood | [21/562] 1209111
0

Loading weights:   0%|          | 0/403 [00:00<?, ?it/s]

07:52:46 | INFO | lmc.mert | [1/562] 65851
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
07:52:56 | INFO | lmc.mert | [2/562] 91431
07:53:06 | INFO | lmc.mert | [3/562] 167539
07:53:31 | INFO | lmc.mert | [4/562] 222237
07:53:43 | INFO | lmc.mert | [5/562] 281140
07:53:55 | INFO | lmc.mert | [6/562] 326218
07:54:04 | INFO | lmc.mert | [7/562] 434739
07:54:15 | INFO | lmc.mert | [8/562] 489995
07:54:23 | INFO | lmc.mert | [9/562] 531194
07:54:35 | INFO | lmc.mert | [10/562] 635544
07:54:47 | INFO | lmc.mert | [11/562] 669831
07:54:59 | INFO | lmc.mert | [12/562] 834390
07:55:14 | INFO | lmc.mert | [13/562] 895865
07:55:22 | INFO | lmc.mert | [14/562] 1003269
07:55:31 | INFO | lmc.mert | [15/562] 1050842
07:55:42 | INFO | lmc.mert | [16/562] 1074484
07:55:54 | INFO | lmc.mert | [17/562] 1147609
07:56:04 | INFO | lmc.mert | [18/562] 1152931
07:56:15 | INFO | lmc.mert | [19/562] 1175444
07:56:26 | INFO | lmc.mert | [20/562] 1196880
07:56:32 | INFO | lmc.mert | 

{'attempted': 1000, 'done': 1000}

## 6 — Embeddings (MuQ-MuLan + LAION-CLAP)

Computes and **caches** a per-song embedding bundle (`data/embeddings/<model>/<id>.npz`)
covering song / line-window / segment levels. Only songs without a cached bundle
are processed. First run downloads model weights.

**CLAP** is loaded via the official `laion_clap` package (the `transformers`
`ClapModel` path silently failed to load this checkpoint's projection weights —
matched audio-text cosine ≈ 0). Download the music checkpoint once and point
`LMC_CLAP_CKPT` at it:

```bash
# https://huggingface.co/lukewys/laion_clap → music_audioset_epoch_15_esc_90.14.pt
export LMC_CLAP_CKPT=/path/to/music_audioset_epoch_15_esc_90.14.pt
```

If you previously cached zero-filled CLAP bundles, re-embed with `force=True`.

In [3]:
embeddings.embed_pending('mulan', limit=None)
embeddings.embed_pending('clap',  limit=None, force=True)   # needs LMC_CLAP_CKPT (music ckpt)
# Re-run after a CLAP fix / version change to overwrite stale bundles:
# embeddings.embed_pending('clap', force=True)

15:17:08 | INFO | lmc.embeddings | Embeddings[mulan]: nothing pending.
15:17:08 | INFO | lmc.embeddings | Embeddings[clap]: 1075 songs (device=mps).
15:17:11 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
15:17:11 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
15:17:11 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
15:17:11 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
15:17:11 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
15:17:11 | INFO | httpx |

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
15:17:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
15:17:14 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary R

Load the specified checkpoint /Users/budge.13/.cache/huggingface/hub/models--lukewys--laion_clap/snapshots/b3708341862f581175dba5c356a4ebf74a9b6651/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...


15:17:15 | INFO | lmc.embeddings | [1/1075] 65851


logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm2.bias 	 Loaded
audio_branch.layers.0.blocks.0

15:17:28 | INFO | lmc.embeddings | [2/1075] 91431
15:17:39 | INFO | lmc.embeddings | [3/1075] 167539
15:17:56 | INFO | lmc.embeddings | [4/1075] 198184
15:18:04 | INFO | lmc.embeddings | [5/1075] 222237
15:18:17 | INFO | lmc.embeddings | [6/1075] 252527
15:18:28 | INFO | lmc.embeddings | [7/1075] 281140
15:18:55 | INFO | lmc.embeddings | [8/1075] 326218
15:19:09 | INFO | lmc.embeddings | [9/1075] 434739
15:19:22 | INFO | lmc.embeddings | [10/1075] 489995
15:19:30 | INFO | lmc.embeddings | [11/1075] 531194
15:19:46 | INFO | lmc.embeddings | [12/1075] 620465
15:19:57 | INFO | lmc.embeddings | [13/1075] 635544
15:20:16 | INFO | lmc.embeddings | [14/1075] 669831
15:20:35 | INFO | lmc.embeddings | [15/1075] 697163
15:20:46 | INFO | lmc.embeddings | [16/1075] 698853
15:21:13 | INFO | lmc.embeddings | [17/1075] 834390
15:21:34 | INFO | lmc.embeddings | [18/1075] 895865
15:21:43 | INFO | lmc.embeddings | [19/1075] 1003269
15:22:05 | INFO | lmc.embeddings | [20/1075] 1050842
15:22:21 | INFO | l

{'attempted': 1075, 'done': 1075}

## 6b — Genre ensemble

Resolve each song's coarse genre by a cascade — **Spotify** artist tags →
**MusicBrainz** recording/artist genres → **zero-shot** from the cached MuLan
embedding (`"This is a {genre} song"`). Records `genre`, `genre_source`, and a
`genre_confidence` per song, shrinking the large `unknown` bin. Runs after
embeddings (zero-shot needs them) and before `combine` (which prefers this over
the Spotify-only genre).

In [4]:
genre.recover_pending()   # cascade: Spotify → MusicBrainz → zero-shot (MuLan)

14:10:59 | INFO | lmc.embeddings | Loading MuQ-MuLan on mps…
14:10:59 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:10:59 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/OpenMuQ/MuQ-MuLan-large/2e01c796b71dca71b45251384c04cd7b237c9020/config.json "HTTP/1.1 200 OK"
14:10:59 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-large-msd-iter/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:10:59 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/OpenMuQ/MuQ-large-msd-iter/0562a57814f6f8bbd9fdea0a25921a2fce1a841a/config.json "HTTP/1.1 200 OK"
/opt/anaconda3/envs/lmc/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
14:11:00 | I

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
14:11:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
14:11:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
14:11:05 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
14:11:05 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/xlm-robert

{'attempted': 1000,
 'spotify': 597,
 'musicbrainz': 109,
 'zeroshot': 161,
 'unknown': 133}

## 7 — Alignment (LMC) & 8 — Combine

Turns cached embeddings into LMC values (song / line-windows / chorus vs.
non-chorus) and assembles the analysis tables in `results/`.

In [5]:
alignment.compute_pending()
combine.build_master()

14:24:11 | INFO | lmc.alignment | LMC[mulan]: scoring 562 songs.
14:24:11 | INFO | lmc.alignment |   [mulan] 25/562
14:24:11 | INFO | lmc.alignment |   [mulan] 50/562
14:24:11 | INFO | lmc.alignment |   [mulan] 75/562
14:24:12 | INFO | lmc.alignment |   [mulan] 100/562
14:24:12 | INFO | lmc.alignment |   [mulan] 125/562
14:24:12 | INFO | lmc.alignment |   [mulan] 150/562
14:24:12 | INFO | lmc.alignment |   [mulan] 175/562
14:24:12 | INFO | lmc.alignment |   [mulan] 200/562
14:24:12 | INFO | lmc.alignment |   [mulan] 225/562
14:24:13 | INFO | lmc.alignment |   [mulan] 250/562
14:24:13 | INFO | lmc.alignment |   [mulan] 275/562
14:24:13 | INFO | lmc.alignment |   [mulan] 300/562
14:24:13 | INFO | lmc.alignment |   [mulan] 325/562
14:24:13 | INFO | lmc.alignment |   [mulan] 350/562
14:24:13 | INFO | lmc.alignment |   [mulan] 375/562
14:24:14 | INFO | lmc.alignment |   [mulan] 400/562
14:24:14 | INFO | lmc.alignment |   [mulan] 425/562
14:24:14 | INFO | lmc.alignment |   [mulan] 450/562
14

{'master_rows': 1550,
 'line_rows': 373406,
 'master_path': '/Users/budge.13/Desktop/Music Analysis/results/master_results.csv'}

In [6]:
import pandas as pd
pd.read_csv(config.RESULTS_DIR / 'master_results.csv').head()

,track_id,title,artist,album,duration,n_synced_lines,spotify_popularity,spotify_id,release_date,deezer_rank,...,mert_pc02,mert_pc03,mert_pc04,mert_pc05,mert_pc06,mert_pc07,mert_pc08,mert_pc09,mert_pc10,song_age_years
0,6783,I Think You Oughta Try Whiskey,Corb Lund feat. Jaida Dreyer,Agricultural Tragic,185.0,43,38.0,1AUh1XSmVhQ66QMrhVIQXk,2020-06-26,125826.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0
1,65851,Hold Me Down,Minus the Bear,Omni,248.0,35,29.0,2ZhjqBX13KwHP9qn0wSEHP,2010-05-04,77904.0,...,9.200316,5.860712,13.164203,2.474716,-4.118803,-2.249334,6.420908,4.632525,2.043223,16.0
2,91431,Time Has Told Me,Nick Drake,Five Leaves Left,264.0,36,53.0,20FLGZPgMHXlU0VpQ0HpxN,1969-07-03,392826.0,...,-10.166129,5.410360,8.515057,-4.311794,1.299915,-5.159216,7.861586,2.383772,-4.432951,57.0
3,167539,If You Let Me Love You,Fleetwood Mac,Boston,630.0,20,16.0,7oKJvdn9ntvU5R2q19zipW,2003,38900.0,...,-4.881096,3.685989,3.256580,1.910034,8.985835,0.550427,3.103149,-3.462920,-4.803062,NaN
4,198184,Dark Side of the Moon (Radio Mix),Ernesto feat. Bastian,Dark Side of the Moon,190.0,22,59.0,5gqGKzCDdxPHnRSkGoxD66,2020-07-09,208679.0,...,10.553439,-2.904990,-0.590255,13.167699,8.797683,0.749334,5.850833,0.980758,0.470688,6.0


## Progress dashboard

Re-run any time to see how far each stage has gotten. Safe to interrupt and
resume: rerun the cells above and only the unfinished songs are processed.

In [7]:
db.progress_report()

{'sampled': 1550,
 'audio_done': 1075,
 'audio_failed': 475,
 'popularity_found': 1546,
 'mood_done': 1075,
 'emb_mulan': 1075,
 'emb_clap': 1075,
 'lmc_rows': 14924}

## Next: statistics & modeling (R)

```bash
Rscript analysis/summary_stats.R              # quick descriptives + figures

# Fit the v4 battery (track measures + segment / curvature / segment+curvature)
# for both embeddings on one shared corpus, with MERT controls:
Rscript stan/run_models.R both 0 42 mert      # args: embeddings, N(0=all), seed, controls

# Evaluate + report:
Rscript stan/evaluate_models.R                # diagnostics, LOO, PPC figures
quarto render analysis/lmc_report.qmd -P embedding:mulan -P controls:mert
```

Control toggle: `controls ∈ {mood, mert, both}` (default `mert`). Re-fit with a
different value and compare LOO to see whether MERT PCs beat the mood proxies.
`run_models.R` also exposes `sample_corpus(df, N, seed)` and `build_corpus()`.